In [1]:
import pandas as pd
import numpy as np

# Carrega o arquivo diretamente da pasta do projeto
df = pd.read_csv(r'C:\Users\LAISA\vibe-bridge-projeto\dataset_clean.csv')
print(f"Dataset carregado com sucesso: {len(df)} faixas.")

Dataset carregado com sucesso: 89740 faixas.


## 0. Feature Selection e Recomendação Híbrida (Acústica 18D + Macro-Famílias Estéticas)

Nesta seção, implementamos a arquitetura de recomendação refinada que combina o melhor de dois mundos: **precisão acústica e coerência estética perceptual**:
1. **Feature Selection e Ponderação via Alvo Proxy (`track_genre`)**: Treinamos um classificador supervisionado (*ExtraTreesClassifier*) sobre 18 variáveis acústicas e harmônicas para derivar os pesos estatísticos ótimos ($W$).
2. **Macro-Famílias Estéticas (Solução para o Abismo Semântico)**: Os 114 gêneros são agrupados em 10 macro-famílias culturais (ex: *Pop & Dance*, *Urban/R&B/Hip-Hop*, *Rock & Alternative*, *Classical & Cinematic*). Calculamos a matriz de afinidade entre famílias para garantir que o modelo permita transições ricas (*Cross-Genre* entre estilos compatíveis), mas evite saltos estéticos dissonantes (como Teen Pop colidindo com Trap Metal agressivo).
3. **Consistência de Cadência Vocal**: Penalidade suave para discrepâncias extremas de estilo vocal (`speechiness`), preservando a harmonia entre faixas melódicas.
4. **Re-ranking Indie (α) e Amostragem Estocástica (Softmax)**: Controle calibrado de popularidade (*Lado B*) e exploração estocástica.

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.cluster import KMeans

# 1. Seleção das variáveis numéricas
features_num = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

X = df[features_num]
y = df['track_genre']

# 2. Escalonamento dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Ponderação do Vetor de Pesos (W)
et_model = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
et_model.fit(X_scaled, y)

weights = et_model.feature_importances_
weights = weights / weights.sum()

# 4. CRIAÇÃO OBRIGATÓRIA DA COLUNA MACRO_CLUSTER
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df['macro_cluster'] = kmeans.fit_predict(X_scaled)
df['macro_family'] = df['macro_cluster']

print("Engenharia de recursos e Macro-Cluster criados com sucesso!")

Engenharia de recursos e Macro-Cluster criados com sucesso!


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

def recomendar_musicas(
    nome_musica,
    nome_artista=None,
    df_base=df,
    feature_matrix=X_scaled,
    family_matrix=None,
    top_k=5,
    alpha_indie=0.0,
    peso_familia=0.20,
    top_m_candidates=50,
    usar_amostragem=False,
    temperatura=1.0,
    filtro_seguranca=True
):
    # Localização da música de consulta
    mask_busca = df_base['track_name'].str.lower().str.contains(nome_musica.lower(), na=False)
    if nome_artista:
        mask_busca &= df_base['artists'].str.lower().str.contains(nome_artista.lower(), na=False)

    resultados_busca = df_base[mask_busca]
    if resultados_busca.empty:
        print(f"❌ Música '{nome_musica}' não encontrada no dataset.")
        return None

    idx_query = resultados_busca.sort_values(by='popularity', ascending=False).index[0]
    musica_query = df_base.iloc[idx_query]
    
    col_family = 'macro_family' if 'macro_family' in df_base.columns else 'macro_cluster'
    fam_query = musica_query[col_family]
    
    print(f"🔍 Consulta: '{musica_query['track_name']}' - {musica_query['artists']} "
          f"[{musica_query['track_genre']} | Família: {fam_query}] "
          f"(Popularidade: {musica_query['popularity']}/100 | BPM: {musica_query['tempo']:.1f})")

    # Similaridade de Cosseno Ponderada
    vetor_query = feature_matrix[idx_query:idx_query+1]
    similaridades = cosine_similarity(vetor_query, feature_matrix).flatten()

    candidatos_df = df_base.copy()
    candidatos_df['sim_acustica'] = similaridades
    candidatos_df = candidatos_df.drop(index=idx_query).drop_duplicates(subset=['track_name', 'artists'])

    # Filtros de Higiene Sonora
    if filtro_seguranca:
        candidatos_df = candidatos_df[
            (candidatos_df['speechiness'] < 0.66) &
            (candidatos_df['duration_min'] >= 1.0)
        ]

    # Afinidade Estética e Cadência Vocal
    candidatos_df['family_affinity'] = (candidatos_df[col_family] == fam_query).astype(float)
    candidatos_df['vocal_diff'] = np.abs(candidatos_df['speechiness'] - musica_query['speechiness'])
    candidatos_df['vocal_consistency'] = 1.0 - np.clip(candidatos_df['vocal_diff'] * 1.5, 0.0, 0.3)

    candidatos_df['sim_ajustada'] = (candidatos_df['sim_acustica'] * candidatos_df['vocal_consistency']) + (candidatos_df['family_affinity'] * peso_familia)

    # Re-ranking Indie
    candidatos_df['anti_popularity'] = 1.0 - (candidatos_df['popularity'] / 100.0)
    candidatos_df['score_final'] = (1.0 - alpha_indie) * candidatos_df['sim_ajustada'] + alpha_indie * candidatos_df['anti_popularity']

    candidatos_top_m = candidatos_df.sort_values(by='score_final', ascending=False).head(top_m_candidates)

    if usar_amostragem and len(candidatos_top_m) >= top_k:
        scores = candidatos_top_m['score_final'].values
        exp_scores = np.exp((scores - np.max(scores)) / temperatura)
        probs = exp_scores / np.sum(exp_scores)

        indices_escolhidos = np.random.choice(candidatos_top_m.index, size=top_k, replace=False, p=probs)
        recomendadas = candidatos_top_m.loc[indices_escolhidos].sort_values(by='score_final', ascending=False)
    else:
        recomendadas = candidatos_top_m.head(top_k)

    return recomendadas[['track_name', 'artists', 'track_genre', col_family, 'popularity', 'tempo', 'energy', 'sim_acustica', 'score_final']]

In [4]:
print("=== DEMO 1: Recomendação Padrão ===")
df_rec1 = recomendar_musicas('Baby', nome_artista='Justin Bieber', alpha_indie=0.0, peso_familia=0.20, top_k=5)
print(df_rec1.to_string(index=False))

print("\n=== DEMO 2: Modo Lado B / Indie ===")
df_rec2 = recomendar_musicas('Baby', nome_artista='Justin Bieber', alpha_indie=0.15, peso_familia=0.20, top_k=5, usar_amostragem=True, temperatura=0.8)
print(df_rec2.to_string(index=False))

=== DEMO 1: Recomendação Padrão ===
🔍 Consulta: 'Baby' - Justin Bieber;Ludacris [pop | Família: 3] (Popularidade: 82/100 | BPM: 65.0)
                             track_name                                artists track_genre  macro_family  popularity  tempo  energy  sim_acustica  score_final
                         Wir eskalieren                        FiNCH;Mia Julia       party             3          30 87.523   0.909      0.956068     1.154634
                                  Face2                              LOZAREENA     j-dance             3          43 86.723   0.856      0.954916     1.144889
                           Miss You Bad                      Mr Eazi;Burna Boy   dancehall             3          54 99.988   0.746      0.924331     1.118785
Burning Love - D-Block & S-te-Fan Remix       Critical Mass;D-Block & S-te-Fan       happy             3          16 75.000   0.746      0.955864     1.112850
                                 Ignite Alan Walker;Julie Bergan;K-391;

## 2. Ordenação Inteligente de Playlists (Smart Playlist Sequencing / DJ Flow) e Avaliação de Fluidez

Nesta seção, implementamos o algoritmo de **Ordenação Inteligente de Playlists (DJ Flow)** e um framework de **Métricas Matemáticas de Fluidez** para avaliar rigorosamente a qualidade das transições musicais:
1. **Grafo de Transição Acústica Ponderada**: Mede o custo de transição $Cost(M_a, M_b) = 1.0 - Sim_{cosseno}(V_a, V_b)$ no espaço de vibe otimizado por Feature Selection ($X_{vibe\_opt}$).
2. **Algoritmo de Vizinho Mais Próximo (*Nearest Neighbor Traversal*)**: Traça o caminho de transição contínua que minimiza quebras bruscas de andamento e energia.
3. **Framework de Avaliação de Fluidez**:
   - **Suavidade Média (%)**: Proximidade harmônica média entre faixas consecutivas normalizada de $0\%$ a $100\%$.
   - **Gargalo Acústico (*Worst Transition*)**: A pior transição individual da playlist, identificando pontos críticos de choque.
   - **Taxa de Choque de BPM ($|\Delta BPM| > 20$)**: Frequência de saltos abruptos no andamento rítmico.
   - **Taxa de Choque de Energia ($|\Delta Energy| > 0.35$)**: Frequência de quebras bruscas de intensidade sonora.

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# =============================================================================
# 1. ALGORITMO DJ FLOW E FRAMEWORK DE AVALIAÇÃO DE FLUIDEZ (CORRIGIDO)
# =============================================================================

def organizar_playlist_suave(indices_playlist, feature_matrix=X_scaled, df_base=df, idx_primeira_musica=None):
    """
    Re-ordena uma playlist do usuário para garantir transições suaves entre faixas consecutivas.
    """
    if len(indices_playlist) <= 2:
        return df_base.iloc[indices_playlist]

    # Mapeia os índices do DataFrame para posições inteiras (0 a N-1) da matriz
    sub_matrix = feature_matrix[indices_playlist]

    # Matriz de Similaridade de Cosseno e Custo de Transição
    sim_matrix = cosine_similarity(sub_matrix)
    cost_matrix = 1.0 - sim_matrix

    nao_visitados = list(range(len(indices_playlist)))

    # Seleção da música de abertura
    if idx_primeira_musica is not None and idx_primeira_musica in indices_playlist:
        start_pos = indices_playlist.index(idx_primeira_musica)
    else:
        # Se não especificada, inicia pela faixa com menor energia (aquecimento gradual)
        energias = df_base.iloc[indices_playlist]['energy'].values
        start_pos = int(np.argmin(energias))

    # Percurso Ganancioso do Vizinho Mais Próximo no Grafo de Transição
    ordem_posicoes = [start_pos]
    nao_visitados.remove(start_pos)

    pos_atual = start_pos
    while nao_visitados:
        proxima_pos = min(nao_visitados, key=lambda pos: cost_matrix[pos_atual, pos])
        ordem_posicoes.append(proxima_pos)
        nao_visitados.remove(proxima_pos)
        pos_atual = proxima_pos

    # Mapeamento para os índices originais do DataFrame
    indices_ordenados = [indices_playlist[pos] for pos in ordem_posicoes]
    df_ordenado = df_base.iloc[indices_ordenados].copy()

    # Cálculo da métrica de suavidade da transição normalizada (0% a 100%)
    suavidades = []
    for i in range(len(ordem_posicoes) - 1):
        p1, p2 = ordem_posicoes[i], ordem_posicoes[i + 1]
        pct_suavidade = ((sim_matrix[p1, p2] + 1.0) / 2.0) * 100.0
        suavidades.append(f"{pct_suavidade:.1f}%")

    df_ordenado['suavidade_transicao'] = suavidades + ['- (Fim da Playlist)']
    return df_ordenado[['track_name', 'artists', 'track_genre', 'tempo', 'energy', 'valence', 'suavidade_transicao']]


def avaliar_fluidez_playlist(df_playlist, feature_matrix=X_scaled, df_base=df):
    """
    Calcula métricas quantitativas de fluidez e transição acústica para uma sequência de músicas.
    """
    # Mapeamento seguro de posições inteiras para fatiar a matriz feature_matrix
    indices_inteiros = [df_base.index.get_loc(idx) for idx in df_playlist.index]
    sub_vecs = feature_matrix[indices_inteiros]

    sims = []
    saltos_bpm = 0
    saltos_energia = 0

    for i in range(len(indices_inteiros) - 1):
        v1, v2 = sub_vecs[i], sub_vecs[i + 1]
        cosseno = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        sims.append(cosseno)

        row1, row2 = df_playlist.iloc[i], df_playlist.iloc[i + 1]
        if abs(row1['tempo'] - row2['tempo']) > 20.0:
            saltos_bpm += 1
        if abs(row1['energy'] - row2['energy']) > 0.35:
            saltos_energia += 1

    n_transicoes = len(indices_inteiros) - 1 if len(indices_inteiros) > 1 else 1
    return {
        'Suavidade Média': f"{((np.mean(sims) + 1.0) / 2.0) * 100.0:.1f}%",
        'Pior Transição (Gargalo)': f"{((np.min(sims) + 1.0) / 2.0) * 100.0:.1f}%",
        'Taxa de Choque de BPM (>20)': f"{(saltos_bpm / n_transicoes) * 100.0:.1f}%",
        'Taxa de Choque de Energia (>0.35)': f"{(saltos_energia / n_transicoes) * 100.0:.1f}%"
    }

# =============================================================================
# 2. DEMONSTRAÇÃO PRÁTICA E COMPARAÇÃO ESTATÍSTICA DE FLUIDEZ
# =============================================================================
np.random.seed(42)
indices_playlist_amostra = df.sample(n=10, random_state=42).index.tolist()
df_bruta = df.loc[indices_playlist_amostra]

print("=== 1. PLAYLIST BRUTA DO USUÁRIO (SEM ORDENAÇÃO) ===")
print(df_bruta[['track_name', 'artists', 'track_genre', 'tempo', 'energy']].to_string(index=False))

# Aplica a ordenação inteligente DJ Flow
df_playlist_opt = organizar_playlist_suave(
    indices_playlist=indices_playlist_amostra,
    feature_matrix=X_scaled,
    df_base=df
)

print("\n=== 2. PLAYLIST RE-ORDENADA PELO ALGORITMO DJ FLOW (TRANSIÇÕES SUAVES) ===")
print(df_playlist_opt.to_string(index=False))

# Avaliação quantitativa de métricas de fluidez
metricas_bruta = avaliar_fluidez_playlist(df_bruta, X_scaled, df)
metricas_flow = avaliar_fluidez_playlist(df_playlist_opt, X_scaled, df)

df_metricas_comp = pd.DataFrame([
    {'Cenário': 'Playlist Bruta (Shuffle / Desordenada)', **metricas_bruta},
    {'Cenário': 'Playlist Otimizada (DJ Flow Acústico)', **metricas_flow}
])

print("\n=== 3. COMPARAÇÃO QUANTITATIVA DE FLUIDEZ ACÚSTICA ===")
print(df_metricas_comp.to_string(index=False))

=== 1. PLAYLIST BRUTA DO USUÁRIO (SEM ORDENAÇÃO) ===
                                        track_name                                artists track_genre   tempo  energy
                                        Girlfriend                             (Hed) P.E.  industrial 179.959  0.6980
                                           Bad Bad                              Roy Woods        soul  87.307  0.5260
                                             別再想見我                                    許光漢    mandopop 141.849  0.4170
                              Bolivia - Radio Edit                          Gunz For Hire   hardstyle 150.016  0.9950
                                    The First Noel                          Gabby Barrett     country  79.993  0.3510
                                          Separate                    Trampled by Turtles   bluegrass 126.286  0.6040
Piano Sonata No. 12 in F Major, K. 332: I. Allegro Wolfgang Amadeus Mozart;Michael Wessel   classical 142.151  0.0743
   